# Validação do ambiente U-Mamba (estágio 11)

Prepara o Google Colab para executar a arquitetura oficial `UMambaEnc_2d` sem compilar o Mamba do zero.

**Antes de executar:** selecione uma GPU NVIDIA (T4 ou outra disponível).

**Importante:** na primeira execução, o notebook pode instalar um stack binário compatível e reiniciar o runtime automaticamente. Quando o Colab reconectar, execute **tudo novamente uma segunda vez**.

## 1. Bootstrap do TCC

Carrega o código mais recente do repositório.

In [ ]:
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## 2. Preparar stack binário do Mamba

Usa PyTorch 2.10 + CUDA 12.8 e uma wheel CUDA pré-compilada do `mamba-ssm`. Isso evita a compilação local que pode levar dezenas de minutos no Colab.

In [ ]:
from src.models.umamba_runtime import install_prebuilt_colab_stack, restart_colab_runtime

restart_required = install_prebuilt_colab_stack()
if restart_required:
    restart_colab_runtime()
else:
    print("Stack binário já está compatível. Nenhum reinício necessário.")


## 3. Confirmar GPU e versões

Depois do reinício, confirme que CUDA e o stack fixado estão ativos.

In [ ]:
import platform
import torch

print(f"Sistema: {platform.platform()}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA do PyTorch: {torch.version.cuda}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA não está ativa no Colab.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## 4. Validar Mamba e preparar U-Mamba oficial

Executa um bloco Mamba na GPU e prepara a arquitetura `UMambaEnc_2d` oficial em um commit fixado.

In [ ]:
import json
from src.models.umamba_runtime import ensure_umamba_runtime

runtime_info = ensure_umamba_runtime()
print(json.dumps(runtime_info, indent=2, ensure_ascii=False))


## 5. Smoke test U-Mamba RGB

Constrói `UMambaEnc_2d` com entrada RGB 256×256 e executa um forward real na GPU.

In [ ]:
from src.config import get_config
from src.models.umamba import build_official_umamba_enc_2d

config = get_config()
features = tuple(int(v) for v in config["model"]["umamba_features"])
device = torch.device("cuda")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = build_official_umamba_enc_2d(
    input_channels=3,
    num_classes=1,
    input_size=(256, 256),
    features_per_stage=features,
).to(device)
x = torch.randn(1, 3, 256, 256, device=device)
with torch.inference_mode():
    y = model(x)

peak_vram_mb = torch.cuda.max_memory_allocated() / 1024**2
parameters = sum(p.numel() for p in model.parameters())
print(f"Entrada: {tuple(x.shape)}")
print(f"Saída: {tuple(y.shape)}")
print(f"Parâmetros: {parameters:,}")
print(f"Pico de VRAM: {peak_vram_mb:.1f} MB")
if tuple(y.shape) != (1, 1, 256, 256):
    raise RuntimeError(f"Shape inesperado: {tuple(y.shape)}")
print("SMOKE TEST U-MAMBA: OK")


## 6. Salvar relatório no Drive


In [ ]:
from src import io

storage_paths = io.resolve_storage_paths()
runs_dir = storage_paths["artifacts_runs"]
runs_dir.mkdir(parents=True, exist_ok=True)
report = {
    **runtime_info,
    "input_shape": list(x.shape),
    "output_shape": list(y.shape),
    "parameters": parameters,
    "smoke_test_peak_vram_mb": peak_vram_mb,
    "status": "ok",
}
report_path = runs_dir / "umamba_environment.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Relatório salvo em: {report_path}")


## Critério para seguir ao notebook 12

Só prossiga quando aparecer **`SMOKE TEST U-MAMBA: OK`**.